# Backtest Engine Template

A creative, modular notebook starter for research-grade strategy testing.

## What this template includes
- Config-driven setup
- Synthetic and CSV market data loaders
- Signal + portfolio + execution separation
- Metrics, tearsheet table, and equity chart
- Parameter sweep scaffold
- Walk-forward analysis scaffold

> Tip: replace the toy strategy with your own alpha logic while keeping the interfaces intact.

In [7]:
from __future__ import annotations

from dataclasses import dataclass
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use("seaborn-v0_8-darkgrid")
pd.options.display.float_format = '{:,.4f}'.format

In [11]:
@dataclass
class BacktestConfig:
    symbol: str = "NVDA"
    frequency: str = "D"
    initial_capital: float = 100_000.0
    fee_bps: float = 1.0
    slippage_bps: float = 2.0
    annualization: int = 252
    seed: int = 42
    period: int = '5y'
    interval: str = '1d'
    # Toy strategy defaults (moving-average crossover)
    fast_window: int = 20
    slow_window: int = 100

cfg = BacktestConfig()
cfg

BacktestConfig(symbol='NVDA', frequency='D', initial_capital=100000.0, fee_bps=1.0, slippage_bps=2.0, annualization=252, seed=42, period='5y', interval='1d', fast_window=20, slow_window=100)

## 1) Data layer

Import stocks data using yahoo finance, and reformatting the Dataframe

In [18]:
def to_ticker_df(df_wide, ticker):
    out = pd.DataFrame({
        'Open':  df_wide['Open'][ticker],
        'High':  df_wide['High'][ticker],
        'Low':   df_wide['Low'][ticker],
        'Close': df_wide['Close'][ticker],
        'Volume': df_wide['Volume'][ticker],
    }).dropna()
    out.index.name = 'Date'
    return out

# You can change the period if you want more/less data
market = yf.download(cfg.symbol, period=cfg.period , interval=cfg.interval, auto_adjust=True, progress=False)

market = to_ticker_df(market, ticker=cfg.symbol)

market.head()


,Open,High,Low,Close,Volume
Date,,,,,
2021-03-01,13.8361,13.8860,13.5152,13.8029,353184000
2021-03-02,13.8610,13.8815,13.3584,13.3687,264116000
2021-03-03,13.3886,13.4138,12.7629,12.7688,377592000
2021-03-04,12.7649,12.9386,12.0499,12.3356,573344000
2021-03-05,12.5148,12.5148,11.6465,12.4266,542840000


In [ ]:
#market.to_csv(f"data/stocks_csv/{cfg.symbol}_data.csv")

## 2) Alpha layer (signals)

Signal convention used in this template:
- +1: fully long
-  0: flat
- -1: fully short

In [20]:
def moving_average_crossover_signal(close: pd.Series, fast: int, slow: int) -> pd.Series:
    fast_ma = close.rolling(fast).mean()
    slow_ma = close.rolling(slow).mean()
    signal = np.where(fast_ma > slow_ma, 1, -1)
    signal = pd.Series(signal, index=close.index, name="target_position")
    signal = signal.where(fast_ma.notna() & slow_ma.notna(), 0)
    return signal

target_position = moving_average_crossover_signal(
    market["Close"],
    fast=cfg.fast_window,
    slow=cfg.slow_window,
)
target_position.tail()

Date
2026-02-23    1
2026-02-24    1
2026-02-25    1
2026-02-26    1
2026-02-27   -1
Name: target_position, dtype: int32

## 3) Execution + portfolio accounting

The engine below keeps things intentionally simple while preserving realistic structure:
- next-bar execution via lagged position
- transaction cost = turnover × (fees + slippage)
- equity curve compounded from daily net returns

In [39]:
def run_backtest(
    market: pd.DataFrame,           # DataFrame containing market data (must include 'close' price)
    target_position: pd.Series,     # Strategy signal series (-1 short, 0 flat, +1 long)
    cfg: BacktestConfig,            # Configuration object (capital, fees, slippage)
) -> pd.DataFrame:

    df = market.copy()  # Copy market data to avoid modifying the original dataset

    # Align strategy signals with market timestamps, fill missing signals with 0 (flat),
    # and limit positions between -1 and 1
    df["target_position"] = target_position.reindex(df.index).fillna(0).clip(-1, 1)

    # Shift position by one bar so trades execute on the next period
    # This prevents look-ahead bias in the backtest
    df["position"] = df["target_position"].shift(1).fillna(0)

    # Calculate percentage return of the asset based on closing prices
    df["asset_return"] = df["Close"].pct_change().fillna(0)

    # Strategy gross return = position exposure * asset return
    # Long earns positive returns, short earns inverse returns
    df["gross_return"] = df["position"] * df["asset_return"]

    # Calculate turnover (how much the position changes each step)
    # Used to estimate trading costs
    turnover = df["position"].diff().abs().fillna(df["position"].abs())

    # Convert trading fees and slippage from basis points to decimal
    total_cost_rate = (cfg.fee_bps + cfg.slippage_bps) / 10_000

    # Trading cost per period = turnover * cost rate
    df["cost_return"] = turnover * total_cost_rate

    # Net return after subtracting transaction costs
    df["net_return"] = df["gross_return"] - df["cost_return"]

    # Calculate equity curve by compounding net returns
    df["equity"] = cfg.initial_capital * (1 + df["net_return"]).cumprod()

    # Track the highest equity value achieved so far
    df["peak_equity"] = df["equity"].cummax()

    # Drawdown measures the percentage drop from the historical peak
    df["drawdown"] = df["equity"] / df["peak_equity"] - 1

    return df  # Return full backtest results DataFrame


# Run the backtest using market data, strategy signals, and config settings
results = run_backtest(market, target_position, cfg)

# Display the last rows showing price, position, portfolio value, and drawdown
results[["Close", "position", "equity", "drawdown"]].tail()

,Close,position,equity,drawdown
Date,,,,
2026-02-23,191.5500,1.0000,"342,648.1445",-0.3427
2026-02-24,192.8500,1.0000,"344,973.6137",-0.3383
2026-02-25,195.5600,1.0000,"349,821.2960",-0.3290
2026-02-26,184.8900,1.0000,"330,734.6083",-0.3656
2026-02-27,177.1900,1.0000,"316,960.7131",-0.3920


## 4) Evaluation metrics

In [ ]:
def summarize_performance(bt: pd.DataFrame, annualization: int = 252) -> pd.Series:
    r = bt["net_return"].dropna()
    eq = bt["equity"]

    total_return = eq.iloc[-1] / eq.iloc[0] - 1
    years = max(len(r) / annualization, 1 / annualization)
    cagr = (1 + total_return) ** (1 / years) - 1

    vol = r.std() * np.sqrt(annualization)
    sharpe = (r.mean() * annualization) / vol if vol > 0 else np.nan

    downside = r[r < 0].std() * np.sqrt(annualization)
    sortino = (r.mean() * annualization) / downside if downside > 0 else np.nan

    max_dd = bt["drawdown"].min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    win_rate = (r > 0).mean()

    stats = pd.Series({
        "Total Return": total_return,
        "CAGR": cagr,
        "Annual Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,
        "Max Drawdown": max_dd,
        "Win Rate": win_rate,
        "Avg Daily Return": r.mean(),
        "Avg Daily Turnover": bt["position"].diff().abs().mean(),
    })
    return stats

summary = summarize_performance(results, annualization=cfg.annualization)
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(results.index, results["equity"], label="Strategy equity", lw=2)
axes[0].set_title(f"{cfg.symbol} Strategy Equity Curve")
axes[0].legend(loc="upper left")

axes[1].fill_between(results.index, results["drawdown"], 0, color="crimson", alpha=0.3)
axes[1].set_title("Drawdown")

plt.tight_layout()
plt.show()

## 5) Parameter sweep scaffold

Grid-search moving-average parameters and rank by Sharpe.

In [ ]:
def parameter_sweep(
    market: pd.DataFrame,
    cfg: BacktestConfig,
    fast_grid: List[int],
    slow_grid: List[int],
) -> pd.DataFrame:
    rows = []
    for fast in fast_grid:
        for slow in slow_grid:
            if fast >= slow:
                continue
            sig = moving_average_crossover_signal(market["close"], fast=fast, slow=slow)
            bt = run_backtest(market, sig, cfg)
            stats = summarize_performance(bt, annualization=cfg.annualization)
            rows.append({
                "fast": fast,
                "slow": slow,
                "Sharpe": stats["Sharpe"],
                "CAGR": stats["CAGR"],
                "Max Drawdown": stats["Max Drawdown"],
            })

    out = pd.DataFrame(rows).sort_values("Sharpe", ascending=False).reset_index(drop=True)
    return out

grid_results = parameter_sweep(market, cfg, fast_grid=[10, 20, 30, 40], slow_grid=[80, 100, 120, 150])
grid_results.head(10)

## 6) Walk-forward analysis scaffold

Use rolling train/test windows to reduce overfitting from static parameter selection.

In [ ]:
def walk_forward_template(
    market: pd.DataFrame,
    cfg: BacktestConfig,
    train_size: int = 500,
    test_size: int = 125,
) -> pd.DataFrame:
    folds = []
    start = 0

    while start + train_size + test_size <= len(market):
        train = market.iloc[start : start + train_size]
        test = market.iloc[start + train_size : start + train_size + test_size]

        # Pick best params from training window
        train_grid = parameter_sweep(
            train,
            cfg,
            fast_grid=[10, 20, 30],
            slow_grid=[80, 100, 120],
        )
        best = train_grid.iloc[0]

        # Evaluate on out-of-sample test window
        sig = moving_average_crossover_signal(test["close"], int(best.fast), int(best.slow))
        bt = run_backtest(test, sig, cfg)
        stats = summarize_performance(bt, annualization=cfg.annualization)

        folds.append({
            "start": test.index.min(),
            "end": test.index.max(),
            "fast": int(best.fast),
            "slow": int(best.slow),
            "Sharpe": stats["Sharpe"],
            "CAGR": stats["CAGR"],
            "Max Drawdown": stats["Max Drawdown"],
        })

        start += test_size

    return pd.DataFrame(folds)

wf_results = walk_forward_template(market, cfg)
wf_results

## 7) Next upgrades

- Multi-asset portfolio construction (risk parity, vol targeting, beta neutral)
- Event-driven engine with order objects and fill simulation
- Benchmark-relative attribution and factor decomposition
- Integration with hyperparameter optimization (Optuna/Ray Tune)
- Production handoff: package into modules and add unit tests